In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
product_category = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [ ]:
print("customers shape", customers.shape)
print("geolocation shape", geolocation.shape)
print("order_items shape", order_items.shape)
print("order_payments shape", order_payments.shape)
print("order_reviews shape", order_reviews.shape)
print("orders shape", orders.shape)
print("products shape", products.shape)
print("product_category shape", product_category.shape)

In [ ]:
customers.info()

In [ ]:
geolocation.head(3)

In [ ]:
orders.info()

In [ ]:
null_rows = orders[orders.isnull().any(axis=1)]
null_rows

In [ ]:
carrier_date_null = orders["order_delivered_customer_date"].isnull().sum()
carrier_date_null

In [ ]:
customer_date_null = orders["order_delivered_carrier_date"].isnull().sum()
customer_date_null

In [ ]:
estimated_delivery_date_null = orders["order_estimated_delivery_date"].isnull().sum()
estimated_delivery_date_null

In [ ]:
order_status_vs_delivered_customer_date = pd.crosstab(orders["order_status"],orders["order_delivered_customer_date"].isna())
order_status_vs_delivered_customer_date

In [ ]:
order_status_vs_order_delivered_carrier_date  = pd.crosstab(orders["order_status"],orders["order_delivered_carrier_date"].isna())
order_status_vs_order_delivered_carrier_date 

In [ ]:
order_status_vs_order_estimated_delivery_date = pd.crosstab(orders["order_status"],orders["order_estimated_delivery_date"].isna())
order_status_vs_order_estimated_delivery_date

In [ ]:
order_status_vs_order_approved_at  = pd.crosstab(orders["order_status"],orders["order_approved_at"].isna())
order_status_vs_order_approved_at 

In [ ]:
# check whether unavailable orders in the orders table are in the orders_items table 
unaivalable_ids = orders.loc[orders["order_status"] == "unavailable","order_id"]
print(len(unaivalable_ids))

In [ ]:
has_items = unaivalable_ids.isin(order_items["order_id"])
print(has_items.sum())

In [ ]:
six_orders = unaivalable_ids[has_items]
order_items[order_items["order_id"].isin(six_orders)]

In [ ]:
orders[orders["order_id"].isin(six_orders)]

In [ ]:
six_orders.nunique()

In [ ]:
# check if the 6 orders are special from the 603
unavail = orders[orders["order_status"] == "unavailable"]
unavail["order_approved_at"].isna().sum()

In [ ]:
orders.info()

In [ ]:
approved_orders = len(orders[orders['order_status'] == "approved"])
delivered_orders = len(orders[orders['order_status'] == "delivered"])
invoiced_orders = len(orders[orders['order_status'] == "invoiced"])
shipped_orders = len(orders[orders['order_status'] == "shipped"])
processing_orders = len(orders[orders['order_status'] == "processing"])
unavailable_orders = len(orders[orders['order_status'] == "unavailable"])
canceled_orders = len(orders[orders['order_status'] == "canceled"])
created_orders = len(orders[orders['order_status'] == "created"])

In [ ]:
orders["order_status"].unique()

In [ ]:
approved_orders+invoiced_orders+shipped_orders+processing_orders+unavailable_orders+canceled_orders+created_orders

In [ ]:
99441-2963

In [ ]:
delivered_orders

In [ ]:
orders["order_status"].value_counts(dropna=False)

In [ ]:
# orders that can be labeleed are the one ones that were delivered and has customer delivery date 
labelable= orders[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].notna())].copy()
len(labelable)

In [ ]:
labelable["order_status"].unique()

In [ ]:
labelable.head(5)

In [ ]:
labelable.dtypes

In [ ]:
# covert both order_delivered_customer_date and order_estimated_delivery_date 
labelable['order_delivered_customer_date'] = pd.to_datetime(labelable['order_delivered_customer_date'])
labelable['order_estimated_delivery_date'] = pd.to_datetime(labelable['order_estimated_delivery_date'])

In [ ]:
labelable.dtypes

In [ ]:
labelable.head(5)

In [ ]:
print(labelable.columns.tolist())

In [ ]:
labelable.dtypes

In [ ]:
labelable.head(5)

In [ ]:
# label for late or not late
labelable["is_late"] = (labelable["order_delivered_customer_date"].dt.normalize() > labelable["order_estimated_delivery_date"]).astype(int)

In [ ]:
print(labelable.columns)

In [ ]:
labelable.head()

In [ ]:
labelable["is_late"].value_counts()

In [ ]:
labelable[labelable["is_late"] == 0]

In [ ]:
labelable["is_late"].isna().sum()

In [ ]:
labelable["order_purchase_timestamp"] = pd.to_datetime(labelable["order_purchase_timestamp"])

In [ ]:
# promised days feature 
labelable["promised_days"] = (labelable["order_estimated_delivery_date"]-labelable["order_purchase_timestamp"].dt.normalize()).dt.days

In [ ]:
labelable.head()

In [ ]:
labelable["promised_days"].isna().sum()

In [ ]:
labelable["promised_days"].describe()

In [ ]:
# do late orders have a longer or a shorter average promised window than on-time orders
labelable.groupby("is_late")["promised_days"].describe()

In [ ]:
# use bins to categorise the promised_days 

In [ ]:
labelable["promised_bucket"] = pd.cut(labelable["promised_days"],bins=[0,10,20,30,40,200])

In [ ]:
labelable.groupby("promised_bucket", observed=True)["is_late"].agg(["mean","count"])

In [ ]:
# promised_days
# - What it is: the estimated delivery date minus the purchase date, in days. It is the delivery window promised to the customer.
# - Why it's safe: the promise is shown to the customer at purchase, so the value exists when the model has to predict. The actual duration (arrival minus purchase) is only known afterwards, so it leaks.
# - What the bins showed: late rate is 10.4% for promises under 10 days (412 late orders) and 2.25% for 40+ days (87 late orders), against 6.8% overall. In the 10 to 40 day range, where most orders sit, rates stay between 5.6% and 7.7%, close to the overall rate.
# - Verdict: weak on its own, because the signal sits at the two ends and the middle shows little. Keep it as a candidate feature and decide later by comparing models with and without it. The pattern is an association and may be driven by distance, category or seller.

In [ ]:
labelable.info()

In [ ]:
order_items.info()

In [ ]:
order_items.head()

In [ ]:
order_features = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    n_sellers=("seller_id", "nunique"),
).reset_index()

In [ ]:
order_features.head()

In [ ]:
len(order_features)

In [ ]:
order_features["order_id"].nunique() 

In [ ]:
order_features["order_id"].is_unique

In [ ]:
modelling = labelable.merge(order_features, on="order_id", how="left")

In [ ]:
# Aggregating order_items
# - Problem: order_items has one row per item, so a plain merge turns 96,470 orders into 110,189 rows and changes the late rate (6.59% vs 6.77%).
# - Fix: group by order_id first (n_items, total_price, total_freight, n_sellers), then left-join onto labelable.
# - Checks: 96,470 rows and 96,470 unique orders after the merge, with no missing values in the new columns.

In [ ]:
len(modelling)

In [ ]:
modelling["is_late"].mean()

In [ ]:
modelling[["n_items","total_price","total_freight","n_sellers"]].isna().sum()

In [ ]:
# how many labelable orders are missing in order_features and vicevarsa
(~labelable["order_id"].isin(order_features["order_id"])).sum()
(~order_features["order_id"].isin(labelable["order_id"])).sum()

In [ ]:
# check number of items and how they relate to lateness
modelling.groupby("n_items")["is_late"].agg(["mean","count"])

In [ ]:
bins=0,1,2,3,np.inf
labels = ["1-2", "2-3", "3-4", "4+"]

In [ ]:
modelling["n_items_bucket"] = pd.cut(modelling["n_items"],bins=bins,labels=labels)

In [ ]:
modelling.head()

In [ ]:
modelling["n_items_bucket"].isna().sum()

In [ ]:
modelling.groupby("n_items_bucket",observed=True)["is_late"].agg(["mean","count"])

In [ ]:
# n_items
# - Late rate: 6.9% for 1 item, 4.9% to 5.4% for 2 to 4+ items (baseline 6.8%).
# - Verdict: weak signal; only about 10% of orders have more than one item. Association only.

In [ ]:
bins2=0,1,np.inf
labels2 = ["1-2", "2+"]

In [ ]:
modelling["n_sellers_bucket"] = pd.cut(modelling["n_sellers"],bins=bins2,labels=labels2)

In [ ]:
modelling.groupby("n_sellers_bucket",observed=True)["is_late"].agg(["mean","count"])

In [ ]:
# compare promise padding to sellers
modelling.groupby("n_sellers_bucket",observed=True)["promised_days"].describe()

In [ ]:
modelling.groupby("n_sellers_bucket",observed=True)["promised_days"].agg(["mean","count"])

In [ ]:
# n_sellers
# - Late rate: 6.85% for 1 seller, about 1% for 2+ (roughly 13 late orders out of 1,275).
# - Promise is about 7% longer for multi-seller orders (26.1 vs 24.4 days), which explains only a small part of the gap.
# - Caveat: several sellers always means several items, so the signal may overlap with n_items. Overlap test not run yet.
# - Verdict: weak candidate, decide after modeling.

In [ ]:
modelling.head()

In [ ]:
modelling.groupby("promised_bucket")["total_freight"].describe()

In [ ]:
modelling["freight_bin"] = pd.qcut(modelling["total_freight"],q=5)

In [ ]:
modelling.head()

In [ ]:
# freight_bin mean for lateness
modelling.groupby("freight_bin", observed=True)["is_late"].agg(["mean","count"])

In [ ]:
# freight_bins lift from the baseline
baseline = 0.0677309

bins = {
    "(-0.001, 12.83]":0.044182,
    "(12.83, 15.64]":0.060442,
    "(15.64, 18.59]":0.076058,
    "(18.59, 27.38]":0.076827,
    "(27.38, 1794.96]":0.081241
}

for b,val in bins.items():
    print(f"{b}:Lift = {val/baseline:.3f}")


In [ ]:
modelling["price_bins"] = pd.qcut(modelling["total_price"],q=5)

In [ ]:
modelling.groupby("price_bins")["is_late"].agg(["count","mean"])

In [ ]:
# price_bins lift from the baseline
baseline = 0.0677309

bins = {
    "(0.849, 39.0]":0.058914,
    "(39.0, 67.9]":0.065596,
    "(67.9, 106.6]":0.066016,
    "(106.6, 175.0]":0.073324,
    "(175.0, 13440.0]":0.074861
}

for b, value in bins.items():
    print(f"{b}: Lift = {value/baseline:.3f}")

In [ ]:
# does the price affect the freight_price
modelling[["n_items","total_price","total_freight"]].corr(method="spearman")

In [ ]:
#  total_freight and total_price (five equal-sized bins by quantile, about 19,300 orders each)

# total_freight: late rate rises steadily from about 4.4% (lowest fifth) to about 8.1% (highest fifth), lift 0.65 to 1.20 against the 6.77% baseline.
# total_price: weaker climb, from about 5.9% to about 7.5%, lift 0.87 to 1.11.
# Correlations (Spearman): price and freight 0.47, freight and n_items 0.38, price and n_items 0.18. No pair is close to a copy, so all are kept.
# Caveat: freight mixes weight, package size and distance, so the mechanism is a hypothesis. Association only.
# Verdict: freight is the strongest feature so far. Price is weaker and partly overlaps with freight. Next: product weight and dimensions, and distance.

In [ ]:
items_products = order_items.merge(products, on="product_id",how="left")
items_products.head()

In [ ]:
missing = items_products[items_products["product_weight_g"].isna()]
missing["product_id"].isin(products["product_id"]).sum()

In [ ]:
missing["order_id"].isin(labelable["order_id"]).sum()

In [ ]:
missing["product_id"].nunique() 

In [ ]:
missing["order_id"].nunique()

In [ ]:
items_products.columns

In [ ]:
items_products.loc[items_products["product_weight_g"]==0, "product_weight_g"]=np.nan

In [ ]:
items_products["product_weight_g"].isna().sum()

In [ ]:
missing[["order_id","product_id"]]

In [ ]:
check_missing = items_products[items_products["order_id"] == "415cfaaaa8cea49f934470548797fed1"]
check_missing[["order_id","product_id","product_weight_g"]]

In [ ]:
weight_features = items_products.groupby("order_id").agg(
    total_weight_g=("product_weight_g", "sum"),
    weight_missing=("product_weight_g", lambda s: s.isna().any()),
).reset_index()

In [ ]:
# total_weight_g (product weight per order)
# - Source: product weight from the products table, joined at the item level, then summed per order.
# - Why it's safe: product weight is known when the customer picks the items, so it exists at purchase time.
# - Missing data: 18 item rows had blank product details (2 products, 16 orders), and 8 item rows had a weight of 0 g, which is not a real weight. All 26 were treated as missing.
# - Handling: a plain sum would skip blanks and give a false total (0 g, or a partial sum). So weight_missing is True when any item in the order lacks a weight, and total_weight_g is set to NaN for those 22 orders.
# - Checks: 96,470 rows after the merge; 22 flagged orders and 22 NaN totals.

In [ ]:
len(weight_features)

In [ ]:
weight_features[weight_features["weight_missing"]== True]

In [ ]:
modelling.head()

In [ ]:
modelling = modelling.drop(columns=["total_weight_g_y","weight_missing_y","total_weight_g_x","weight_missing_x","total_weight_g","weight_missing"])

In [ ]:
modelling.head()

In [ ]:
modelling = modelling.merge(weight_features,on="order_id",how="left")

In [ ]:
modelling.loc[modelling["weight_missing"] == True, "total_weight_g"] = np.nan

In [ ]:
len(modelling)

In [ ]:
modelling["weight_missing"].sum()

In [ ]:
modelling["total_weight_g"].isna().sum()

In [ ]:
modelling.head()

In [ ]:
modelling.drop(columns=["weight_bins"])

In [ ]:
modelling["weight_bins"] = pd.qcut(modelling["total_weight_g"],q=5)

In [ ]:
modelling.groupby("weight_bins")["is_late"].agg(["mean","count"])

In [ ]:
# weight_bins lift from the baseline
baseline = 0.0677309

bins = {
    "(-0.001, 250.0]":0.060835,
    "(250.0, 500.0]":0.070209,
    "(500.0, 1100.0]":0.062451,
    "(1100.0, 2800.0]":0.070292,
    "(2800.0, 184400.0]":0.075453
}

for b, value in bins.items():
    print(f"{b}: Lift = {value/baseline:.3f}")

In [ ]:
(modelling["total_weight_g"] == 0).sum()

In [ ]:
(items_products["product_weight_g"] == 0).sum()

In [ ]:
modelling.head()

In [ ]:
modelling[["total_weight_g","total_freight","total_price"]].corr(method="spearman")

In [ ]:
# Late rate by weight and correlations
# - Late rate across five equal-sized weight bins (about 19,000 orders each) zigzags between 6.1% and 7.5% (lift 0.90 to 1.11 against the 6.8% baseline). The heaviest fifth is highest, but the pattern is not steady.
# - My prediction was that the late rate would rise proportionally with weight. It did not, so weight is a weaker signal than freight.
# - Spearman correlations: weight and price 0.54, weight and freight 0.49, price and freight 0.47. All moderate, so no feature is a copy of another and all are kept for now.
# - Weight and freight are only moderately related, which suggests freight also reflects something other than weight, possibly distance. This is a hypothesis to test with the distance feature.
# - Verdict: weak candidate. Association only.

In [ ]:
weight_features["weight_missing"].sum()

In [ ]:
modelling["total_weight_g"].isna().sum()